# 🧠 UnblurNews — DistilBERT Multi-Task Training

Fine-tunes **distilbert-base-uncased** on three simultaneous classification tasks:

| Head | Task | Classes |
|------|------|---------|
| `clickbait_head` | Clickbait detection | 0 = Not clickbait · 1 = Clickbait |
| `leaning_head` | Political leaning | 0 = Left · 1 = Center · 2 = Right |
| `sentiment_head` | Sentiment | 0 = Negative · 1 = Neutral · 2 = Positive |

**Runtime:** GPU → T4 (Runtime → Change runtime type → GPU)  
**Expected total time:** ~30–45 min on T4  
**Output:** `unblur_model.zip` — ready to drop into `unblur/backend/models/`

In [1]:
# @title ⚙️ Install Dependencies
%%capture
!pip install transformers datasets accelerate scikit-learn tqdm pandas huggingface_hub

In [2]:
# @title 📦 Imports & Configuration
import os, time, math, random, json, zipfile, shutil
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader, random_split
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, classification_report
from tqdm.auto import tqdm

# Mixed-precision — use the non-deprecated API (PyTorch >= 2.0)
try:
    from torch.amp import autocast, GradScaler
    AMP_DEVICE = "cuda"
except ImportError:
    from torch.cuda.amp import autocast, GradScaler
    AMP_DEVICE = "cuda"

# ── Reproducibility ──────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ── Device ───────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    torch.cuda.manual_seed(SEED)
    print(f"GPU:    {torch.cuda.get_device_name(0)}")
    print(f"VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU — training on CPU will be very slow.")
    print("    Go to: Runtime → Change runtime type → GPU (T4)")

# ── Hyperparameters ───────────────────────────────────────────────
MODEL_NAME      = "distilbert-base-uncased"   # 66M params, ~2x faster than BERT-base
MAX_LEN_SHORT   = 128    # clickbait & sentiment (headlines / tweets)
MAX_LEN_LONG    = 256    # political leaning (article paragraphs)
BATCH_SIZE      = 32     # reduce to 16 if you get CUDA out-of-memory
LR              = 2e-5
WEIGHT_DECAY    = 0.01
WARMUP_RATIO    = 0.06
MAX_GRAD_NORM   = 1.0
EPOCHS_TASK     = 2      # epochs per individual-head training phase
EPOCHS_MULTI    = 1      # epochs for the final multi-task phase
MAX_SAMPLES     = 8_000  # cap per task — keeps total runtime ~30-45 min on T4
USE_FP16        = device.type == "cuda"

# ── Paths ─────────────────────────────────────────────────────────
OUTPUT_DIR      = "/content/unblur_model"
CKPT_DIR        = "/content/checkpoints"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(CKPT_DIR).mkdir(parents=True, exist_ok=True)

print(f"\nConfig: model={MODEL_NAME} | batch={BATCH_SIZE} | fp16={USE_FP16}")
print(f"Output: {OUTPUT_DIR}")

Device: cuda
GPU:    Tesla T4
VRAM:   15.6 GB

Config: model=distilbert-base-uncased | batch=32 | fp16=True
Output: /content/unblur_model


## 📊 Section 1 — Load Datasets

Each loader tries multiple sources and falls back gracefully.

In [3]:
# @title 📰 1A — Clickbait Dataset
from datasets import load_dataset

def load_clickbait(max_samples=MAX_SAMPLES):
    print("Loading clickbait data...")

    SOURCES = [
        # (dataset_id, text_col, label_col)
        ("christinacdl/clickbait_notclickbait_dataset", "text",  "label"),
        ("marksverdhei/clickbait_title_classification",  "title", "clickbait"),
        ("rkotari/clickbait",                            "text",  "label"),
    ]

    texts, labels = [], []
    for ds_id, text_col, label_col in SOURCES:
        print(f"  Trying {ds_id}...")
        try:
            ds = load_dataset(ds_id, trust_remote_code=True)
            for split in ds.values():
                raw_labels = [str(l) for l in split[label_col]]
                # normalise string labels ("clickbait"/"factual") to int
                if raw_labels[0] in ("clickbait", "factual", "0", "1"):
                    raw_labels = [1 if l in ("1", "clickbait") else 0 for l in raw_labels]
                else:
                    raw_labels = [int(l) for l in raw_labels]
                texts  += [str(t) for t in split[text_col]]
                labels += raw_labels
            print(f"  ✓ Loaded {len(texts)} samples")
            break
        except Exception as e:
            print(f"  ✗ Failed: {e}")

    if not texts:
        raise RuntimeError("All clickbait dataset sources failed. Check your internet connection.")

    # Balance & cap
    half = (max_samples or len(texts)) // 2
    pos  = [(t, 1) for t, l in zip(texts, labels) if l == 1][:half]
    neg  = [(t, 0) for t, l in zip(texts, labels) if l == 0][:half]
    data_pairs = pos + neg
    random.shuffle(data_pairs)
    texts, labels = zip(*data_pairs)
    texts, labels = list(texts), list(labels)

    print(f"  Final: {len(texts)} samples  |  clickbait={labels.count(1)}  normal={labels.count(0)}")
    return texts, labels

cb_texts, cb_labels = load_clickbait()

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'christinacdl/clickbait_notclickbait_dataset' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'christinacdl/clickbait_notclickbait_dataset' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading clickbait data...
  Trying christinacdl/clickbait_notclickbait_dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/437 [00:00<?, ?B/s]

train.json: 0.00B [00:00, ?B/s]

...

test.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/43802 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2191 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/8760 [00:00<?, ? examples/s]

  ✓ Loaded 54753 samples
  Final: 8000 samples  |  clickbait=4000  normal=4000


In [4]:
# @title 🗳️ 1B — Political Leaning Dataset  (multiple sources with fallback)
# Labels: 0=Left  1=Center  2=Right

# ── Label string → integer mapping ──────────────────────────────
LEAN_MAP = {
    "left":0,"left-center":0,"lean left":0,"lean-left":0,"liberal":0,
    "progressive":0,"left wing":0,"leftwing":0,"left_center":0,
    "center":1,"centre":1,"neutral":1,"moderate":1,"centrist":1,
    "center-left":1,"center-right":1,"center left":1,"center right":1,"balanced":1,
    "right":2,"right-center":2,"lean right":2,"lean-right":2,"conservative":2,
    "republican":2,"right wing":2,"rightwing":2,"right_center":2,
}

def _map_lean_labels(raw):
    out = []
    for r in raw:
        k = str(r).lower().strip()
        if k in LEAN_MAP:            out.append(LEAN_MAP[k])
        elif k.isdigit() and int(k) in (0,1,2): out.append(int(k))
        else: return None   # unmappable
    return out

TEXT_COLS  = ["text","article","content","headline","title","sentence","body","statement"]
LABEL_COLS = ["label","bias","leaning","category","class","political_leaning",
              "media_bias","orientation","political_bias","bias_label"]

def _try_hf(name, extra_configs=None):
    """Try loading a HuggingFace dataset with flexible column detection."""
    configs = extra_configs or [None]
    for cfg in configs:
        try:
            ds = load_dataset(name, cfg, trust_remote_code=True) if cfg else load_dataset(name, trust_remote_code=True)
            # Merge all splits
            texts, labels = [], []
            for sname, split in ds.items():
                cols = split.column_names
                tc = next((c for c in TEXT_COLS  if c in cols), None)
                lc = next((c for c in LABEL_COLS if c in cols), None)
                if not tc or not lc:
                    continue
                raw = [str(r) for r in split[lc]]
                mapped = _map_lean_labels(raw)
                if mapped is None:
                    unique = set(raw[:20])
                    print(f"    [{name}] Unknown labels: {unique}")
                    continue
                texts  += [str(t) for t in split[tc]]
                labels += mapped
            if len(texts) >= 200:
                return texts, labels
        except Exception as e:
            pass
    return None, None

# ── Fallback (100 examples/class — enough for a functional classifier) ──
FALLBACK_LEFT = [
    "Republican tax cuts for the wealthy are devastating working families and widening inequality.",
    "Universal healthcare is a human right and the government must act to guarantee it for all.",
    "The GOP's refusal to address climate change is negligence against future generations.",
    "Corporate greed and union-busting are the root causes of wage stagnation for workers.",
    "Systemic racism embedded in our institutions demands bold progressive policy reform.",
    "Billionaires should pay their fair share — a wealth tax is long overdue in America.",
    "Workers' rights are under attack by Republican governors pushing anti-union legislation.",
    "Medicare for All would save millions of lives and trillions of dollars over the long term.",
    "The fossil fuel industry has spent decades lying about climate change to protect profits.",
    "Student debt cancellation would lift millions out of poverty and stimulate the economy.",
    "Reproductive rights are human rights — abortion must remain safe, legal, and accessible.",
    "The minimum wage must be raised to $15 or higher to reflect the true cost of living.",
    "Dark money in politics is corrupting our democracy and must be abolished immediately.",
    "Voter suppression laws are designed to disenfranchise Black and brown Americans.",
    "The war on drugs has been a racist policy that devastated communities of color.",
    "Child poverty can be ended with expanded social safety net programs — the will is lacking.",
    "Public education is underfunded while charter schools siphon money from communities.",
    "Gun violence is a public health crisis requiring bold legislative action from Congress.",
    "Trickle-down economics has never worked and has only enriched the billionaire class.",
    "We must abolish the Electoral College and move to a true national popular vote.",
    "Immigrants built this country and xenophobic policies hurt our economy and values.",
    "The prison-industrial complex must be dismantled and replaced with rehabilitation.",
    "Corporate media is a mouthpiece for the wealthy elite, not the working people.",
    "Big Pharma's profit motive kills people — we need price controls on prescription drugs.",
    "The climate crisis demands a Green New Deal and massive public investment in clean energy.",
    "Republicans are waging a war on voting rights to stay in power against the will of voters.",
    "The wealthy buy elections while working people struggle to survive under the current system.",
    "Progressive taxation is the only way to reverse decades of rising wealth inequality.",
    "Healthcare should not be a privilege for the rich — it is a right for every American.",
    "Corporate lobbyists write the laws that protect their profits at the expense of workers.",
    "The gender pay gap is real, persistent, and must be addressed with strong legislation now.",
    "Police brutality against Black Americans is a systemic crisis requiring structural reform.",
    "The Supreme Court has been captured by right-wing ideologues undermining democracy.",
    "Union membership must be expanded to restore the balance of power between labor and capital.",
    "The ultra-wealthy pay lower effective tax rates than teachers and nurses — that must change.",
    "Mass incarceration is a form of modern-day slavery that targets communities of color.",
    "Fossil fuel subsidies must end immediately and be redirected to renewable energy investment.",
    "Every child deserves high-quality public education regardless of their zip code.",
    "The opioid crisis was manufactured by pharmaceutical companies driven by profit over people.",
    "Expanding Medicaid in every state would save lives and reduce financial suffering.",
    "The gig economy exploits workers by denying them benefits and basic labor protections.",
    "Social Security and Medicare must be expanded, not cut, to protect seniors from poverty.",
    "Housing is a human right — rent control and tenant protections must be strengthened now.",
    "The military budget dwarfs social spending — our priorities as a nation are deeply wrong.",
    "Indigenous land rights must be honored and protected against corporate exploitation.",
    "LGBTQ+ Americans deserve full equality under the law without exception or compromise.",
    "The student loan crisis is the result of decades of defunding public higher education.",
    "Wall Street caused the financial crisis and paid no real price — that accountability gap must close.",
    "Environmental racism places toxic facilities disproportionately in communities of color.",
    "Paid family leave is standard in every wealthy nation except the United States — that is shameful.",
    "Republicans have repeatedly tried to strip healthcare from millions of Americans.",
    "The minimum wage has not kept pace with inflation, leaving workers further behind every year.",
    "Gerrymandering by Republicans is an assault on representative democracy.",
    "Citizens United must be overturned to get dark money out of our political system.",
    "The wealth of three billionaires equals that of the bottom half of Americans — that is obscene.",
    "Universal pre-K would be transformative for child development and economic equality.",
    "Criminal justice reform must address the root causes of crime, not just punishment.",
    "The US has the highest rate of child poverty among wealthy nations — that is a political choice.",
    "Worker cooperatives offer a democratic alternative to exploitative corporate structures.",
    "The Green New Deal would create millions of good-paying jobs while saving the planet.",
    "Austerity politics have failed — public investment in people and infrastructure is needed.",
    "Closing the racial wealth gap requires targeted, race-conscious policy, not colorblindness.",
    "The opioid epidemic disproportionately devastated rural white working-class communities ignored by elites.",
    "Single-payer healthcare would eliminate bureaucratic waste and cover every American.",
    "Tax the rich to fund education, healthcare, and infrastructure for everyone.",
    "The Democratic Party must embrace bold progressive policies to win back working-class voters.",
    "Climate change is an existential threat that demands immediate, transformative government action.",
    "Expanding the child tax credit would cut child poverty in half — Republicans killed it.",
    "Corporate consolidation has killed competition and raised prices for ordinary Americans.",
    "The right to organize a union must be protected and strengthened with federal legislation.",
    "America's infrastructure crisis is the result of decades of tax cuts starving public investment.",
    "The student debt crisis represents a failure of public policy that must be corrected.",
    "Defunding the police means reinvesting in mental health, housing, and education services.",
    "The US carceral system is cruel, expensive, and ineffective — it needs radical reform.",
    "Reparations for slavery are a moral and economic imperative that must be seriously considered.",
    "Big tech monopolies are destroying competition and threatening democracy itself.",
    "The for-profit healthcare system puts shareholder returns above patients' lives.",
    "Universal basic income could eliminate poverty and provide security in an automated economy.",
    "Closing offshore tax havens would raise hundreds of billions to invest in public goods.",
    "The war in Iraq was built on lies that cost trillions and countless lives with no accountability.",
    "Federal investment in clean energy will create jobs and reduce carbon emissions simultaneously.",
    "SNAP and food stamps keep millions of children from going hungry — they must be protected.",
    "The gender wealth gap is compounded by race — intersectional policy solutions are needed.",
    "Private prisons create a perverse financial incentive to incarcerate more people.",
    "Raising the capital gains tax to match ordinary income tax rates is basic fairness.",
    "The United States needs a livable planet more than it needs another generation of fossil fuels.",
    "Democrats must fight for working people, not corporate donors who fund both parties.",
    "Early childhood education investment has among the highest returns of any public spending.",
    "The US spends more on healthcare than any other country but has worse outcomes — the system is broken.",
    "Homeownership has been systematically denied to Black Americans through redlining and racism.",
    "Assault weapons have no place in a civilian society — a ban is long overdue.",
    "Medicaid expansion in red states has saved lives and reduced hospital costs dramatically.",
    "The prison population would drop dramatically if addiction were treated as a health issue.",
    "Corporate tax avoidance costs the US hundreds of billions every year in lost public revenue.",
    "Paid sick leave should be a federal right, not a benefit reserved for salaried workers.",
    "The American Dream is broken for most — structural change is needed, not individual hustle.",
    "Big Oil lobbied for decades to block climate action and must be held legally accountable.",
    "Voting rights are under unprecedented attack — federal protection is urgently needed.",
    "The ultra-wealthy have rigged the system in their favor and it is time for the rest of us to fight back.",
]

FALLBACK_CENTER = [
    "The Federal Reserve held interest rates steady, citing stable inflation and moderate growth.",
    "Both parties failed to reach a compromise on the budget, leaving the issue unresolved.",
    "Economists are divided on the long-term effects of the proposed trade policy changes.",
    "The bipartisan infrastructure bill passed with support from members of both parties.",
    "Analysts say the economy shows mixed signals, with job growth offset by rising prices.",
    "Negotiators from both parties are working toward a compromise on immigration legislation.",
    "Independent voters remain skeptical of proposals from both the left and the right.",
    "The nonpartisan budget office projected modest growth under the current fiscal policy.",
    "Representatives from across the aisle expressed concern about rising national debt.",
    "The court's decision drew criticism from both progressive and conservative groups.",
    "Experts say addressing climate change requires realistic policy, not ideological extremism.",
    "Polling shows Americans are frustrated with gridlock in Washington from both parties.",
    "The technology company faced scrutiny from regulators appointed by both parties.",
    "Healthcare reform requires addressing costs while maintaining access — a difficult balance.",
    "Both candidates have proposed competing economic plans with significant trade-offs.",
    "Civil liberties groups expressed concern about surveillance legislation backed by both parties.",
    "The trade agreement has supporters and critics across the political spectrum.",
    "Infrastructure investment enjoys broad public support regardless of political affiliation.",
    "Fiscal responsibility and social investment are both important goals for policymakers.",
    "Campaign finance reform has advocates in both the Democratic and Republican parties.",
    "The unemployment rate fell slightly, though analysts caution against over-interpreting.",
    "The report called for evidence-based immigration policy rather than partisan rhetoric.",
    "The study found that policies from both administrations had pros and cons for growth.",
    "Experts say the most effective criminal justice policies combine accountability and rehab.",
    "The commission released its report calling for reforms to address concerns from both sides.",
    "The Senate passed the measure with votes from both Democrats and Republicans.",
    "Analysts warn that both proposed tax plans carry significant fiscal risks over the long run.",
    "The governor signed the compromise bill after months of negotiations between both caucuses.",
    "Voters in the swing district split their tickets, backing candidates from both major parties.",
    "The nonpartisan watchdog found problems with campaign finance practices on both sides.",
    "A new poll finds that most Americans want cooperation between the two parties on spending.",
    "The former official served under administrations of both political parties over his career.",
    "The independent commission recommended a balanced approach to pension reform.",
    "Both liberal and conservative economists expressed reservations about the proposal.",
    "The legislation passed with broad support after amendments addressed concerns from both sides.",
    "The mayor, a moderate, won support from voters across party lines in Tuesday's election.",
    "Experts from across the political spectrum agree that the current system needs reform.",
    "The committee heard testimony from witnesses representing a wide range of viewpoints.",
    "The report balanced concerns about civil liberties with the need for national security.",
    "The bipartisan caucus released a joint statement urging compromise on the debt ceiling.",
    "Both the left and right have valid concerns about the concentration of power in tech companies.",
    "The independent auditor found no evidence of widespread fraud but noted process gaps.",
    "The governor's budget drew both praise and criticism from legislators on both sides of the aisle.",
    "Centrist voters may prove decisive in November as both parties struggle to expand their bases.",
    "The nonpartisan think tank published a cost-benefit analysis of the proposed healthcare changes.",
    "The judiciary committee questioned the nominee on issues of concern to both parties.",
    "Urban and rural representatives found common ground on agricultural funding in the new bill.",
    "The panel of experts was divided on whether the regulation would achieve its intended goals.",
    "The diplomats from both nations agreed to continue talks aimed at reducing trade tensions.",
    "The proposal has drawn supporters and opponents from across the ideological spectrum.",
    "A compromise was reached on immigration after weeks of negotiations involving both parties.",
    "The report found that economic growth under both recent administrations was broadly similar.",
    "The joint committee issued recommendations drawing on evidence rather than ideology.",
    "Both progressives and conservatives have expressed support for criminal justice reform.",
    "The election results reflected deep divisions but also some areas of cross-partisan agreement.",
    "Experts urge policymakers to set aside partisanship and focus on evidence-based solutions.",
    "The new trade deal has won endorsements from business and labor groups on both sides.",
    "The city council approved the zoning changes with votes from members of both parties.",
    "The task force recommended a phased approach that addressed concerns from all stakeholders.",
    "Most voters polled said they preferred compromise over holding firm on partisan priorities.",
    "The Federal Election Commission released its report citing violations by members of both parties.",
    "Both liberal and conservative legal scholars questioned the constitutional basis of the order.",
    "The independent review board concluded that the agency had made errors under both administrations.",
    "The university study found support for both market-based and regulatory approaches to the problem.",
    "The new defense secretary pledged to work with Congress members from both chambers and parties.",
    "Fiscal hawks from both sides of the aisle expressed alarm over the projected deficit increase.",
    "The town hall drew residents with a wide range of opinions on the proposed development.",
    "The think tank's report called for a pragmatic, evidence-based approach to drug policy.",
    "The candidate positioned herself as a pragmatic problem-solver rather than an ideologue.",
    "The centrist coalition unveiled a healthcare proposal designed to win votes from both parties.",
    "The governor vetoed the bill, citing concerns shared by members of both parties.",
    "Pew Research found that fewer Americans than ever identify strongly with either major party.",
    "The panel recommended a middle-ground approach to immigration that addressed both security and humanitarian concerns.",
    "The independent candidate polled strongly among voters dissatisfied with both major parties.",
    "Both the House and Senate versions of the bill have strengths and weaknesses, analysts say.",
    "The ambassador called for diplomacy and dialogue rather than escalation on both sides.",
    "The school board voted to adopt a curriculum that acknowledged multiple perspectives on history.",
    "The new speaker pledged to run a more inclusive process and seek input from both sides.",
    "The congressional watchdog cited waste and inefficiency in programs run under both parties.",
    "A cross-partisan group of senators introduced a bill to address the opioid crisis.",
    "The city's centrist mayor secured a second term by focusing on practical local governance.",
    "Voters across party lines expressed frustration with the lack of progress on housing affordability.",
    "The summit brought together labor leaders, business groups, and community organizations to find common ground.",
    "The study found that healthcare costs have risen under both Republican and Democratic policies.",
    "The commission called for reforms to restore public trust in institutions across the board.",
    "The bipartisan working group identified several areas where compromise legislation is achievable.",
    "Both parties' candidates have made the economy the central focus of their campaigns.",
    "The international observers praised the election as generally free and fair with minor irregularities.",
    "The nonpartisan legal foundation filed briefs challenging overreach from both the executive and legislative branches.",
    "A former Republican and a former Democrat co-authored the op-ed calling for ranked-choice voting.",
    "The joint task force on infrastructure proposed funding mechanisms drawing from both tax increases and savings.",
    "The new poll showed that most Americans support compromise on immigration regardless of their party affiliation.",
    "The senator called on both sides to lower the temperature and focus on shared goals.",
    "The bipartisan group toured the border together to develop a shared understanding of the situation.",
    "The journalist interviewed officials, activists, and residents from across the political spectrum for her report.",
    "The auditors found that federal programs had been managed with varying degrees of effectiveness under successive administrations.",
    "The economics professor argued that both austerity and stimulus have appropriate uses depending on circumstances.",
    "The commission recommended changes to campaign finance law that addressed concerns raised by both parties.",
]

FALLBACK_RIGHT = [
    "Biden's open-border policies have created an unprecedented national security crisis.",
    "The radical left's socialist agenda is destroying the American economy and way of life.",
    "Tax cuts for job creators stimulate growth, create jobs, and ultimately benefit all Americans.",
    "Second Amendment rights must be protected against the government's unconstitutional overreach.",
    "The mainstream media is covering up the truth — the corrupt Democrat machine must be stopped.",
    "Critical race theory is indoctrinating our children with anti-American propaganda in schools.",
    "Law enforcement deserves our full support — defunding the police endangers every community.",
    "America-first trade policies protect our workers from China's economic and military warfare.",
    "The radical green agenda will destroy our energy independence and kill millions of jobs.",
    "Parents have the right to know what is being taught to their children — no more woke ideology.",
    "Election integrity is non-negotiable — we must ensure every legal vote is counted.",
    "Free market solutions — not government mandates — are the answer to rising healthcare costs.",
    "The Constitution is the supreme law of the land and must be defended against leftist attacks.",
    "America's energy independence was achieved under Republican leadership and must be preserved.",
    "Woke corporations are pushing an extreme political agenda on unwilling American consumers.",
    "The weaponization of the DOJ against conservatives is an unprecedented abuse of power.",
    "Religious liberty is under attack from the radical secularist agenda of the Democrat Party.",
    "Strong borders and controlled immigration protect American workers and communities.",
    "Inflation is the direct result of reckless Democrat spending that must be reversed now.",
    "School choice gives every child — regardless of zip code — access to a quality education.",
    "The military has been weakened by woke ideology when it should be focused on combat readiness.",
    "Judicial activism by left-wing judges is bypassing the will of the American people.",
    "Small businesses are being crushed by excessive government regulation and taxation.",
    "Traditional family values are the foundation of a strong, prosperous, and free society.",
    "Government dependency is destroying communities — we need to restore personal responsibility.",
    "The Democrat Party has been taken over by radical socialists who despise American values.",
    "Crime has skyrocketed in Democrat-run cities because of soft-on-crime progressive prosecutors.",
    "America was founded on Judeo-Christian values that must be defended against the secular left.",
    "Illegal immigration costs American taxpayers billions and threatens public safety.",
    "The radical left wants to disarm law-abiding Americans and leave them defenseless.",
    "Republicans must stand firm against the far-left agenda that is tearing America apart.",
    "The deep state is working to undermine conservative governance and silence dissenting voices.",
    "Excessive government regulation is strangling American business and destroying jobs.",
    "The left's assault on free speech in schools and universities is un-American and dangerous.",
    "Border security is national security — a strong wall and tough enforcement are essential.",
    "The media's coordinated attacks on conservatives reveal their deep political bias.",
    "Capital punishment for the most heinous crimes is justice — not vengeance.",
    "The left's embrace of gender ideology is confusing children and undermining parental authority.",
    "American workers are being replaced by illegal immigrants who drive down wages and take jobs.",
    "The radical environmentalists want to destroy the American energy sector and eliminate jobs.",
    "Conservative values of faith, family, and freedom are the bedrock of American greatness.",
    "The Democrat Party's push for socialism will bankrupt America and destroy individual liberty.",
    "Our brave police officers deserve support, not persecution by left-wing activists.",
    "Parental rights must be restored — schools are indoctrinating children without parents' knowledge.",
    "The left-wing assault on the fossil fuel industry is killing jobs and raising energy prices.",
    "America needs strong leadership to stand up to China, not appeasement from the left.",
    "Election security measures are common-sense protections against fraud, not voter suppression.",
    "The Second Amendment is a God-given right that no government can lawfully take away.",
    "Republican tax cuts unleashed economic growth and created record-low unemployment.",
    "The mainstream media is the propaganda arm of the Democrat Party — trust them at your peril.",
    "Cancel culture is a left-wing weapon to silence conservative voices and punish dissent.",
    "The southern border invasion is a Democrat-created crisis that threatens American sovereignty.",
    "Crime will only be reduced when we hold criminals accountable with tough sentencing.",
    "The left-wing judicial appointments are a threat to the Constitution and the rule of law.",
    "America must rebuild its military strength after years of Democrat-led weakness.",
    "The radical left's climate agenda would destroy the American economy for no measurable benefit.",
    "Defunding the police was a disastrous experiment that led to skyrocketing murder rates.",
    "Republicans must protect American jobs from China's predatory trade and theft of technology.",
    "The woke takeover of corporate America is an attack on conservative consumers and workers.",
    "School choice and education savings accounts empower parents and improve student outcomes.",
    "The only way to stop inflation is to cut government spending and stop the socialist handouts.",
    "America's founders never intended for the government to have this much power over our lives.",
    "The left has turned our universities into indoctrination factories hostile to conservative thought.",
    "A strong economy requires low taxes, less regulation, and the freedom to innovate and compete.",
    "The Biden administration's energy policies are a surrender to radical environmentalists.",
    "Law and order must be restored in American cities destroyed by progressive governance.",
    "The Democrat Party has abandoned working-class Americans in favor of radical coastal elites.",
    "Strong families, not government programs, are the answer to poverty and social breakdown.",
    "The mainstream media refuses to cover stories that make Democrats look bad.",
    "American sovereignty demands secure borders — anyone who disagrees is not serious about governing.",
    "Tax and spend policies have never worked and never will — limited government is the answer.",
    "The woke military is endangering national security by prioritizing diversity over lethality.",
    "Republicans believe in the dignity of work, not the dependency of endless government welfare.",
    "Big tech censorship of conservative voices is a clear and present danger to free speech.",
    "The left-wing indoctrination of children must be stopped — parents must take back control.",
    "America's greatness rests on individual liberty, not collective redistribution of wealth.",
    "The Democrat agenda of open borders and soft-on-crime policies is destroying American cities.",
    "Restoring American manufacturing requires standing up to China's economic aggression.",
    "The assault on the police has made every community in America less safe.",
    "Conservative policies of deregulation and tax cuts delivered the strongest economy in decades.",
    "The radical trans agenda being pushed on children is child abuse and must be stopped.",
    "America first means protecting American workers, American jobs, and American sovereignty.",
    "The national debt is a ticking time bomb created by decades of Democrat overspending.",
    "Religious Americans have the right to live and work according to their deeply held beliefs.",
    "The left wants to replace the American dream with a socialist nightmare — we must stop them.",
    "Voter ID is common sense — every other democracy requires it and America should too.",
    "The radical green new deal would eliminate air travel, ban gas stoves, and destroy the economy.",
    "Conservative judges interpret the Constitution as written, not as the left wishes it to be.",
    "The Democrat Party is the party of crime, open borders, and radical socialism — full stop.",
    "Protecting the unborn is the defining moral issue of our time and Republicans stand for life.",
    "The free market, not government, is the engine of prosperity and innovation in America.",
    "Republican leadership produced energy independence — Biden threw it away on day one.",
    "The deep state bureaucracy works overtime to undermine the will of the American voter.",
    "CRT teaches children to hate their country and each other based on the color of their skin.",
    "The constitutional right to bear arms is the ultimate check on government tyranny.",
    "Securing the border is the first responsibility of any government — Democrats have failed.",
    "American values of hard work, personal responsibility, and faith made this nation great.",
]

def load_political(max_samples=MAX_SAMPLES):
    print("Loading political leaning data...")
    texts, labels = None, None

    # Ordered list of datasets to try
    attempts = [
        ("cajcodes/political-news-dataset", None),
        ("valurank/political-news-article-bias", None),
        ("newsmediabias/BEAD", None),
        ("ARTeLab/loco", None),
    ]
    for name, cfg in attempts:
        print(f"  Trying {name}...")
        t, l = _try_hf(name, [cfg] if cfg else None)
        if t and len(t) >= 200:
            print(f"  ✓ Loaded {len(t)} samples from {name}")
            texts, labels = t, l
            break
        else:
            print(f"  ✗ Not usable")

    if texts is None:
        print("\n  ⚠️  All remote datasets failed.")
        print("  Using built-in fallback (100 examples/class).")
        texts  = FALLBACK_LEFT + FALLBACK_CENTER + FALLBACK_RIGHT
        labels = [0]*100 + [1]*100 + [2]*100

    # Balance & cap
    per_class = (max_samples or len(texts)) // 3
    result = []
    for cls in (0, 1, 2):
        pairs = [(t, l) for t, l in zip(texts, labels) if l == cls][:per_class]
        result.extend(pairs)
    random.shuffle(result)
    texts, labels = zip(*result)
    texts, labels = list(texts), list(labels)

    c = {0: labels.count(0), 1: labels.count(1), 2: labels.count(2)}
    print(f"  Final: {len(texts)} samples  |  Left={c[0]}  Center={c[1]}  Right={c[2]}")
    return texts, labels

pol_texts, pol_labels = load_political()

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'cajcodes/political-news-dataset' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'cajcodes/political-news-dataset' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading political leaning data...
  Trying cajcodes/political-news-dataset...


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'valurank/political-news-article-bias' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'valurank/political-news-article-bias' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


  ✗ Not usable
  Trying valurank/political-news-article-bias...


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'newsmediabias/BEAD' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'newsmediabias/BEAD' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'ARTeLab/loco' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not sup

  ✗ Not usable
  Trying newsmediabias/BEAD...
  ✗ Not usable
  Trying ARTeLab/loco...
  ✗ Not usable

  ⚠️  All remote datasets failed.
  Using built-in fallback (100 examples/class).
  Final: 294 samples  |  Left=100  Center=100  Right=94


In [5]:
# @title 😐 1C — Sentiment Dataset  (cardiffnlp/tweet_eval)
# Labels: 0=Negative  1=Neutral  2=Positive

def load_sentiment(max_samples=MAX_SAMPLES):
    print("Loading sentiment data...")
    try:
        ds = load_dataset("cardiffnlp/tweet_eval", "sentiment", trust_remote_code=True)
        texts, labels = [], []
        for sname in ("train", "validation", "test"):
            if sname in ds:
                texts  += [str(t) for t in ds[sname]["text"]]
                labels += [int(l) for l in ds[sname]["label"]]
        print(f"  ✓ tweet_eval/sentiment: {len(texts)} samples")
    except Exception as e:
        print(f"  ✗ tweet_eval failed: {e}")
        print("  Trying amazon_polarity as fallback...")
        ds = load_dataset("amazon_polarity", split="train[:40000]", trust_remote_code=True)
        texts  = [str(t) for t in ds["content"]]
        # 0=negative→0, 1=positive→2 (no neutral class)
        labels = [0 if l == 0 else 2 for l in ds["label"]]
        print(f"  ✓ amazon_polarity fallback: {len(texts)} samples (binary, no neutral)")

    # Balance & cap
    per_class = (max_samples or len(texts)) // 3
    result = []
    for cls in (0, 1, 2):
        pairs = [(t, l) for t, l in zip(texts, labels) if l == cls][:per_class]
        result.extend(pairs)
    random.shuffle(result)
    texts, labels = zip(*result)
    texts, labels = list(texts), list(labels)

    c = {0: labels.count(0), 1: labels.count(1), 2: labels.count(2)}
    print(f"  Final: {len(texts)} samples  |  Neg={c[0]}  Neu={c[1]}  Pos={c[2]}")
    return texts, labels

sent_texts, sent_labels = load_sentiment()

print("\n" + "="*50)
print("Dataset summary")
print("="*50)
print(f"  Clickbait : {len(cb_texts):>6,} samples")
print(f"  Political : {len(pol_texts):>6,} samples")
print(f"  Sentiment : {len(sent_texts):>6,} samples")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'cardiffnlp/tweet_eval' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'cardiffnlp/tweet_eval' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading sentiment data...


README.md: 0.00B [00:00, ?B/s]

sentiment/train-00000-of-00001.parquet:   0%|          | 0.00/3.78M [00:00<?, ?B/s]

sentiment/test-00000-of-00001.parquet:   0%|          | 0.00/901k [00:00<?, ?B/s]

sentiment/validation-00000-of-00001.parq(…):   0%|          | 0.00/167k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/45615 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/12284 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

  ✓ tweet_eval/sentiment: 59899 samples
  Final: 7998 samples  |  Neg=2666  Neu=2666  Pos=2666

Dataset summary
  Clickbait :  8,000 samples
  Political :    294 samples
  Sentiment :  7,998 samples


## 🔧 Section 2 — Preprocessing & DataLoaders

In [6]:
# @title Load tokenizer and build DataLoaders

class TextDataset(Dataset):
    """Pre-tokenizes all texts at init for faster training."""
    def __init__(self, texts, labels, tokenizer, max_length):
        self.encodings = tokenizer(
            texts,
            max_length=max_length,
            padding="max_length",
            truncation=True,
        )
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids":      torch.tensor(self.encodings["input_ids"][idx],      dtype=torch.long),
            "attention_mask": torch.tensor(self.encodings["attention_mask"][idx], dtype=torch.long),
            "labels":         torch.tensor(self.labels[idx],                      dtype=torch.long),
        }

def make_loaders(texts, labels, tokenizer, max_length, train_ratio=0.85):
    """Build balanced train/val DataLoaders."""
    ds = TextDataset(texts, labels, tokenizer, max_length)
    n_train = int(len(ds) * train_ratio)
    n_val   = len(ds) - n_train
    train_ds, val_ds = random_split(ds, [n_train, n_val],
                                    generator=torch.Generator().manual_seed(SEED))
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,   shuffle=True,
                              num_workers=2, pin_memory=(device.type=="cuda"))
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE*2, shuffle=False,
                              num_workers=2, pin_memory=(device.type=="cuda"))
    return train_loader, val_loader

# ── Load tokenizer ────────────────────────────────────────────────
print(f"Loading tokenizer: {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("✓ Tokenizer ready")

# ── Build DataLoaders ─────────────────────────────────────────────
print("\nTokenizing datasets (may take a few minutes)...")
print("  Clickbait...", end="", flush=True)
cb_train, cb_val     = make_loaders(cb_texts,   cb_labels,   tokenizer, MAX_LEN_SHORT)
print(f" {len(cb_train)} train batches")

print("  Political...", end="", flush=True)
pol_train, pol_val   = make_loaders(pol_texts,  pol_labels,  tokenizer, MAX_LEN_LONG)
print(f" {len(pol_train)} train batches")

print("  Sentiment...", end="", flush=True)
sent_train, sent_val = make_loaders(sent_texts, sent_labels, tokenizer, MAX_LEN_SHORT)
print(f" {len(sent_train)} train batches")

print("\n✓ All DataLoaders ready")

Loading tokenizer: distilbert-base-uncased ...


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✓ Tokenizer ready

Tokenizing datasets (may take a few minutes)...
  Clickbait... 213 train batches
  Political... 8 train batches
  Sentiment... 213 train batches

✓ All DataLoaders ready


## 🧠 Section 3 — Model Architecture

Single ModernBERT backbone shared by three classification heads.

In [ ]:
# @title MultiHeadDistilBERT model definition

class MultiHeadModernBERT(nn.Module):
    """
    DistilBERT backbone + three classification heads.

    Head architecture (shared pattern):
        Dropout → Linear(768→256) → ReLU → Dropout → Linear(256→n_classes)

    Head names match unblur/backend/analyzer.py:
        clickbait_head  (2 classes)
        leaning_head    (3 classes: left / center / right)
        sentiment_head  (3 classes: neg / neu / pos)
    """

    def __init__(self, model_name=MODEL_NAME, dropout=0.1):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden = self.backbone.config.hidden_size  # 768 for distilbert-base-uncased

        def _head(n_classes):
            return nn.Sequential(
                nn.Dropout(dropout),
                nn.Linear(hidden, 256),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(256, n_classes),
            )

        self.clickbait_head = _head(2)
        self.leaning_head   = _head(3)   # political leaning
        self.sentiment_head = _head(3)

    def _cls(self, input_ids, attention_mask):
        """Run backbone and return [CLS] token representation."""
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        return out.last_hidden_state[:, 0, :]  # shape: (batch, hidden_size)

    def forward(self, input_ids, attention_mask, task="all"):
        cls = self._cls(input_ids, attention_mask)
        if task == "clickbait":
            return self.clickbait_head(cls)
        elif task == "leaning":
            return self.leaning_head(cls)
        elif task == "sentiment":
            return self.sentiment_head(cls)
        elif task == "all":
            return {
                "clickbait": self.clickbait_head(cls),
                "leaning":   self.leaning_head(cls),
                "sentiment": self.sentiment_head(cls),
            }
        raise ValueError(f"Unknown task: {task!r}")

    def save_ckpt(self, path):
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        torch.save({
            "model_state_dict": self.state_dict(),
            "config": {"model_name": MODEL_NAME},
        }, path)

    @classmethod
    def load_ckpt(cls, path):
        ckpt  = torch.load(path, map_location=device, weights_only=False)
        model = cls(model_name=ckpt["config"].get("model_name", MODEL_NAME))
        model.load_state_dict(ckpt["model_state_dict"])
        return model.to(device)

print(f"Downloading backbone: {MODEL_NAME} ...")
model = MultiHeadModernBERT().to(device)

total = sum(p.numel() for p in model.parameters())
train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✓ Model ready  |  total params: {total/1e6:.1f}M  |  trainable: {train/1e6:.1f}M")

## 🏋️ Section 4 — Training

**Strategy:** Train each head sequentially (backbone updates each time), then joint multi-task fine-tuning.

Mixed-precision (fp16) is used automatically when a GPU is available.

In [8]:
# @title Training utilities (train_epoch, evaluate, train_task)

criterion = nn.CrossEntropyLoss()

def train_epoch(model, loader, optimizer, scheduler, task, scaler):
    model.train()
    total_loss, all_preds, all_true = 0.0, [], []

    pbar = tqdm(loader, desc=f"  [{task}] train", leave=False, ncols=90)
    for batch in pbar:
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        lbls = batch["labels"].to(device)

        optimizer.zero_grad(set_to_none=True)

        if scaler:
            with autocast(AMP_DEVICE):
                logits = model(ids, mask, task=task)
                loss   = criterion(logits, lbls)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(ids, mask, task=task)
            loss   = criterion(logits, lbls)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()

        scheduler.step()
        total_loss += loss.item()
        all_preds  += logits.argmax(dim=-1).cpu().tolist()
        all_true   += lbls.cpu().tolist()
        pbar.set_postfix(loss=f"{loss.item():.3f}")

    return total_loss / len(loader), accuracy_score(all_true, all_preds)


@torch.no_grad()
def evaluate(model, loader, task):
    model.eval()
    total_loss, all_preds, all_true = 0.0, [], []

    for batch in loader:
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        lbls = batch["labels"].to(device)

        if USE_FP16:
            with autocast(AMP_DEVICE):
                logits = model(ids, mask, task=task)
        else:
            logits = model(ids, mask, task=task)

        total_loss += criterion(logits, lbls).item()
        all_preds  += logits.argmax(dim=-1).cpu().tolist()
        all_true   += lbls.cpu().tolist()

    return total_loss / len(loader), accuracy_score(all_true, all_preds), all_preds, all_true


def train_task(model, task, train_loader, val_loader, n_epochs=EPOCHS_TASK):
    """Full training loop for one task. Returns model with best val accuracy."""
    print(f"\n{'='*55}")
    print(f"  Training: {task.upper()} HEAD  ({n_epochs} epochs)")
    print(f"{'='*55}")

    optimizer    = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    total_steps  = len(train_loader) * n_epochs
    warmup_steps = int(total_steps * WARMUP_RATIO)
    scheduler    = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    scaler       = GradScaler(AMP_DEVICE) if USE_FP16 else None

    best_acc  = 0.0
    best_path = f"{CKPT_DIR}/best_{task}.pt"

    for epoch in range(1, n_epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc   = train_epoch(model, train_loader, optimizer, scheduler, task, scaler)
        vl_loss, vl_acc, _, _ = evaluate(model, val_loader, task)
        mins = (time.time() - t0) / 60

        marker = " ★" if vl_acc > best_acc else ""
        print(f"  Epoch {epoch}/{n_epochs}  "
              f"train {tr_loss:.4f}/{tr_acc:.2%}  "
              f"val {vl_loss:.4f}/{vl_acc:.2%}  "
              f"{mins:.1f}m{marker}")

        if vl_acc > best_acc:
            best_acc = vl_acc
            model.save_ckpt(best_path)

    print(f"\n  Best val accuracy: {best_acc:.2%} — reloading checkpoint...")
    return MultiHeadModernBERT.load_ckpt(best_path), best_acc

print("✓ Training utilities defined")

✓ Training utilities defined


### 4A — Clickbait Head
Trains the full model (backbone + clickbait head) for `EPOCHS_TASK` epochs.

In [9]:
# @title Train clickbait head
model, cb_best = train_task(model, "clickbait", cb_train, cb_val)


  Training: CLICKBAIT HEAD  (2 epochs)


  [clickbait] train:   0%|                                        | 0/213 [00:00<?, ?it/s]

  Epoch 1/2  train 0.3626/84.81%  val 0.2846/88.83%  0.4m ★


  [clickbait] train:   0%|                                        | 0/213 [00:00<?, ?it/s]

  Epoch 2/2  train 0.2386/90.91%  val 0.2774/88.25%  0.4m

  Best val accuracy: 88.83% — reloading checkpoint...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### 4B — Political Leaning Head
Continues from the clickbait checkpoint — backbone has already adapted to news text.

In [10]:
# @title Train political leaning head
model, pol_best = train_task(model, "leaning", pol_train, pol_val)


  Training: LEANING HEAD  (2 epochs)


  [leaning] train:   0%|                                            | 0/8 [00:00<?, ?it/s]

  Epoch 1/2  train 1.0490/55.02%  val 0.9785/66.67%  0.0m ★


  [leaning] train:   0%|                                            | 0/8 [00:00<?, ?it/s]

  Epoch 2/2  train 0.9558/71.89%  val 0.9355/77.78%  0.0m ★

  Best val accuracy: 77.78% — reloading checkpoint...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### 4C — Sentiment Head

In [11]:
# @title Train sentiment head
model, sent_best = train_task(model, "sentiment", sent_train, sent_val)


  Training: SENTIMENT HEAD  (2 epochs)


  [sentiment] train:   0%|                                        | 0/213 [00:00<?, ?it/s]

  Epoch 1/2  train 0.8799/58.44%  val 0.7527/67.25%  0.4m ★


  [sentiment] train:   0%|                                        | 0/213 [00:00<?, ?it/s]

  Epoch 2/2  train 0.6474/73.46%  val 0.7378/69.00%  0.4m ★

  Best val accuracy: 69.00% — reloading checkpoint...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### 4D — Multi-Task Fine-Tuning
Trains all three heads simultaneously with a weighted combined loss.
Uses half the learning rate to avoid catastrophic forgetting.

In [12]:
# @title Multi-task fine-tuning

def train_multitask(model, n_epochs=EPOCHS_MULTI):
    print(f"\n{'='*55}")
    print(f"  MULTI-TASK FINE-TUNING  ({n_epochs} epochs, lr={LR/2:.0e})")
    print(f"{'='*55}")

    optimizer    = AdamW(model.parameters(), lr=LR/2, weight_decay=WEIGHT_DECAY)
    steps_epoch  = max(len(cb_train), len(pol_train), len(sent_train))
    total_steps  = steps_epoch * n_epochs
    scheduler    = get_linear_schedule_with_warmup(optimizer,
                       int(total_steps * WARMUP_RATIO), total_steps)
    scaler       = GradScaler(AMP_DEVICE) if USE_FP16 else None

    best_combined = 0.0
    best_path = f"{CKPT_DIR}/best_multitask.pt"

    def _get(it, loader):
        try:    return next(it)
        except: return next(iter(loader))

    for epoch in range(1, n_epochs + 1):
        model.train()
        t0          = time.time()
        total_loss  = 0.0
        cb_it   = iter(cb_train)
        pol_it  = iter(pol_train)
        sent_it = iter(sent_train)

        pbar = tqdm(range(steps_epoch),
                    desc=f"  Multi-task epoch {epoch}/{n_epochs}", leave=False, ncols=90)
        for _ in pbar:
            optimizer.zero_grad(set_to_none=True)

            def fwd(batch, task):
                ids  = batch["input_ids"].to(device)
                mask = batch["attention_mask"].to(device)
                lbls = batch["labels"].to(device)
                return criterion(model(ids, mask, task=task), lbls)

            cb_b, pol_b, sent_b = (_get(cb_it, cb_train),
                                   _get(pol_it, pol_train),
                                   _get(sent_it, sent_train))

            if scaler:
                with autocast(AMP_DEVICE):
                    loss = fwd(cb_b,"clickbait") + fwd(pol_b,"leaning") + fwd(sent_b,"sentiment")
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                scaler.step(optimizer); scaler.update()
            else:
                loss = fwd(cb_b,"clickbait") + fwd(pol_b,"leaning") + fwd(sent_b,"sentiment")
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                optimizer.step()

            scheduler.step()
            total_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.3f}")

        _, cb_acc,   _, _ = evaluate(model, cb_val,   "clickbait")
        _, pol_acc,  _, _ = evaluate(model, pol_val,  "leaning")
        _, sent_acc, _, _ = evaluate(model, sent_val, "sentiment")
        combined = (cb_acc + pol_acc + sent_acc) / 3
        mins = (time.time() - t0) / 60

        marker = " ★" if combined > best_combined else ""
        print(f"  Epoch {epoch}/{n_epochs}  loss={total_loss/steps_epoch:.4f}  "
              f"CB={cb_acc:.2%}  Lean={pol_acc:.2%}  Sent={sent_acc:.2%}  "
              f"avg={combined:.2%}  {mins:.1f}m{marker}")

        if combined > best_combined:
            best_combined = combined
            model.save_ckpt(best_path)

    print(f"\n  Best combined accuracy: {best_combined:.2%} — reloading checkpoint...")
    return MultiHeadModernBERT.load_ckpt(best_path), best_combined

model, multi_best = train_multitask(model)


  MULTI-TASK FINE-TUNING  (1 epochs, lr=1e-05)


  Multi-task epoch 1/1:   0%|                                     | 0/213 [00:00<?, ?it/s]

  Epoch 1/1  loss=1.4645  CB=88.08%  Lean=80.00%  Sent=68.50%  avg=78.86%  1.7m ★

  Best combined accuracy: 78.86% — reloading checkpoint...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 📈 Section 5 — Evaluation

In [13]:
# @title Final evaluation on validation sets

TASK_META = {
    "clickbait": {"loader": cb_val,   "names": ["Not Clickbait", "Clickbait"]},
    "leaning":   {"loader": pol_val,  "names": ["Left", "Center", "Right"]},
    "sentiment": {"loader": sent_val, "names": ["Negative", "Neutral", "Positive"]},
}

print("=" * 60)
print("FINAL EVALUATION")
print("=" * 60)

results = {}
for task, meta in TASK_META.items():
    _, acc, preds, true = evaluate(model, meta["loader"], task)
    results[task] = acc
    print(f"\n─── {task.upper()}  (val accuracy: {acc:.2%}) ───")
    print(classification_report(true, preds, target_names=meta["names"], digits=3))

print("=" * 60)
print(f"Summary: clickbait={results['clickbait']:.2%}  "
      f"leaning={results['leaning']:.2%}  "
      f"sentiment={results['sentiment']:.2%}")
print(f"Average: {sum(results.values())/3:.2%}")

# Quick sanity check with real sentences
print("\n" + "=" * 60)
print("SANITY CHECK — manual inference")
print("=" * 60)
TESTS = [
    ("You WON'T BELIEVE what this politician just said!",
     "Experts react as shocking video goes viral nationwide."),
    ("Federal Reserve holds rates steady amid moderate growth",
     "The central bank cited stable inflation data in its decision Wednesday."),
    ("GOP tax cuts for billionaires devastate working families",
     "Republicans push regressive policy that widens wealth inequality."),
    ("Biden's open-border agenda threatens national security",
     "Conservative critics slam the administration's immigration failures."),
]
model.eval()
with torch.no_grad():
    for title, body in TESTS:
        text = f"{title} [SEP] {body}"
        enc  = tokenizer(text, return_tensors="pt",
                         max_length=256, padding="max_length", truncation=True)
        ids  = enc["input_ids"].to(device)
        mask = enc["attention_mask"].to(device)

        out = model(ids, mask, task="all")
        cb_p  = torch.softmax(out["clickbait"], 1)[0, 1].item() * 100
        lean  = torch.softmax(out["leaning"],   1)[0]
        sent  = torch.softmax(out["sentiment"], 1)[0]
        pol_s = (-1*lean[0] + 0*lean[1] + 1*lean[2]).item()
        sen_s = (-1*sent[0] + 0*sent[1] + 1*sent[2]).item()

        print(f"\nTitle: '{title[:60]}'")
        print(f"  Clickbait: {cb_p:.1f}%  |  Political: {pol_s:+.3f}  |  Sentiment: {sen_s:+.3f}")

FINAL EVALUATION

─── CLICKBAIT  (val accuracy: 88.08%) ───
               precision    recall  f1-score   support

Not Clickbait      0.870     0.892     0.881       595
    Clickbait      0.892     0.869     0.880       605

     accuracy                          0.881      1200
    macro avg      0.881     0.881     0.881      1200
 weighted avg      0.881     0.881     0.881      1200


─── LEANING  (val accuracy: 80.00%) ───
              precision    recall  f1-score   support

        Left      0.857     0.632     0.727        19
      Center      0.933     1.000     0.966        14
       Right      0.625     0.833     0.714        12

    accuracy                          0.800        45
   macro avg      0.805     0.822     0.802        45
weighted avg      0.819     0.800     0.798        45


─── SENTIMENT  (val accuracy: 68.50%) ───
              precision    recall  f1-score   support

    Negative      0.727     0.757     0.742       412
     Neutral      0.587     0.584

## 💾 Section 6 — Export & Download

Saves the model in two formats:
1. **HuggingFace format** backbone + `task_heads.pt` — used by `unblur/backend/analyzer.py`
2. **Full checkpoint** `model_full.pt` — for easy reloading / further training

In [14]:
# @title Export model files

def export_model(model, tokenizer, out_dir=OUTPUT_DIR):
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    model.eval()

    print("Exporting model...\n")

    # 1 — Backbone in HuggingFace format (config.json + model.safetensors)
    print("[1/4] Saving backbone (HuggingFace format)...")
    model.backbone.save_pretrained(str(out))
    print(f"      config.json + model.safetensors → {out}/")

    # 2 — Tokenizer
    print("[2/4] Saving tokenizer...")
    tokenizer.save_pretrained(str(out))
    print(f"      tokenizer.json + vocab files → {out}/")

    # 3 — Task heads only (used by analyzer.py's Format 1)
    print("[3/4] Saving task heads (task_heads.pt)...")
    heads_state = {k: v for k, v in model.state_dict().items()
                   if not k.startswith("backbone.")}
    torch.save(heads_state, out / "task_heads.pt")
    print(f"      {len(heads_state)} tensors saved")
    print(f"      Keys: {list(heads_state.keys())[:4]} ...")

    # 4 — Full checkpoint (used by analyzer.py's Format 2)
    print("[4/4] Saving full checkpoint (model_full.pt)...")
    model.save_ckpt(str(out / "model_full.pt"))

    # Model card
    results_str = "\n".join(f"- {k}: {v:.2%}" for k, v in results.items())
    readme = f"""# UnBlur ModernBERT — Multi-Task Model

Fine-tuned from `{MODEL_NAME}`.

## Validation Accuracy
{results_str}

## Files
- `config.json` + `model.safetensors` — backbone weights (HuggingFace format)
- `tokenizer.json` + tokenizer files  — tokenizer
- `task_heads.pt`  — classification head weights only
- `model_full.pt`  — full model checkpoint (backbone + heads)

## Usage
Copy this folder to `unblur/backend/models/` and start the backend.
"""
    (out / "README.md").write_text(readme)

    # File sizes
    print("\nFile sizes:")
    total_mb = 0
    for f in sorted(out.iterdir()):
        if f.is_file():
            mb = f.stat().st_size / 1e6
            total_mb += mb
            print(f"  {f.name:<40} {mb:6.1f} MB")
    print(f"  {'TOTAL':<40} {total_mb:6.1f} MB")
    print(f"\n✓ Export complete: {out}")

export_model(model, tokenizer)

Exporting model...

[1/4] Saving backbone (HuggingFace format)...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

      config.json + model.safetensors → /content/unblur_model/
[2/4] Saving tokenizer...
      tokenizer.json + vocab files → /content/unblur_model/
[3/4] Saving task heads (task_heads.pt)...
      12 tensors saved
      Keys: ['clickbait_head.1.weight', 'clickbait_head.1.bias', 'clickbait_head.4.weight', 'clickbait_head.4.bias'] ...
[4/4] Saving full checkpoint (model_full.pt)...

File sizes:
  README.md                                   0.0 MB
  config.json                                 0.0 MB
  model.safetensors                         265.5 MB
  model_full.pt                             267.9 MB
  task_heads.pt                               2.4 MB
  tokenizer.json                              0.7 MB
  tokenizer_config.json                       0.0 MB
  TOTAL                                     536.4 MB

✓ Export complete: /content/unblur_model


In [15]:
# @title 📦 Create ZIP and download

ZIP_NAME = "unblur_model.zip"
ZIP_PATH = f"/content/{ZIP_NAME}"

print(f"Creating {ZIP_NAME} ...")
out = Path(OUTPUT_DIR)

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for f in sorted(out.rglob("*")):
        if f.is_file():
            arcname = f"unblur_model/{f.relative_to(out)}"
            zf.write(f, arcname)
            print(f"  + {arcname}")

zip_mb = Path(ZIP_PATH).stat().st_size / 1e6
print(f"\n✓ {ZIP_NAME}  ({zip_mb:.0f} MB)")

print("\nStarting download...")
from google.colab import files
files.download(ZIP_PATH)

print("""
╔══════════════════════════════════════════════════════╗
║  ✓ Download complete!                                ║
║                                                      ║
║  Next steps:                                         ║
║  1. Unzip unblur_model.zip                          ║
║  2. Copy contents to unblur/backend/models/          ║
║  3. Run: uvicorn backend.main:app --reload           ║
╚══════════════════════════════════════════════════════╝
""")

Creating unblur_model.zip ...
  + unblur_model/README.md
  + unblur_model/config.json
  + unblur_model/model.safetensors
  + unblur_model/model_full.pt
  + unblur_model/task_heads.pt
  + unblur_model/tokenizer.json
  + unblur_model/tokenizer_config.json

✓ unblur_model.zip  (494 MB)

Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


╔══════════════════════════════════════════════════════╗
║  ✓ Download complete!                                ║
║                                                      ║
║  Next steps:                                         ║
║  1. Unzip unblur_model.zip                          ║
║  2. Copy contents to unblur/backend/models/          ║
║  3. Run: uvicorn backend.main:app --reload           ║
╚══════════════════════════════════════════════════════╝

